<pre>
- Dióxido de nitrogênio   -> Unidade de medida de retorno kg/kg-¹
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.

Nota de conversão: Na prática e em estudos de qualidade do ar, costuma-se converter kg/kg para fração em volume em partes por milhão (ppm) ou partes por bilhão (ppb) utilizando a massa molar do gás e do ar seco (≈28,96 g/mol)  

In [10]:
import cdsapi
import os, sys
import xarray as xr
import zipfile
from pyspark.sql import functions as F

In [6]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
dataset = "cams-global-reanalysis-eac4"
request = {
    "variable": [
        "carbon_monoxide"
    ],
    "pressure_level": ["1000"],
    "date": ["2025-12-01/2025-12-31"],
    "time": ["06:00"],
    "data_format": "netcdf",
    "area": [6, -74, -35, -34]
}

client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

ret_download = client.retrieve(dataset, request).download()

print(f"Download completed: {ret_download}")

2026-07-22 14:00:05,450 INFO Request ID is bd038406-82ba-4afb-bab5-2ebf553b0036
2026-07-22 14:00:06,507 INFO status has been updated to accepted
2026-07-22 14:00:30,504 INFO status has been updated to running
2026-07-22 14:00:42,103 INFO status has been updated to successful
                                                                                     

Download completed: 434661e1c850d30dbc5a36b1409d862.nc


In [16]:
with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Poluicao\\{ret_download}"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:
    # Transforma o Dataset em um Spark Dataframe
    df_dask             = ds.to_dask_dataframe()
    df_dask_c           = df_dask.compute()
    df_monoxido_carbono = spark.createDataFrame(df_dask_c)

df_monoxido_carbono.printSchema()
df_monoxido_carbono.show(10, False)

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- valid_time: timestamp (nullable = true)
 |-- pressure_level: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- co: float (nullable = true)

+-------------------+--------------+--------+---------+-------------+
|valid_time         |pressure_level|latitude|longitude|co           |
+-------------------+--------------+--------+---------+-------------+
|2025-12-01 06:00:00|1000.0        |6.0     |-73.5    |2.442264E-7  |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.75   |2.0894143E-7 |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.0    |1.609915E-7  |
|2025-12-01 06:00:00|1000.0        |6.0     |-71.25   |1.3711863E-7 |
|2025-12-01 06:00:00|1000.0        |6.0     |-70.5    |1.18243214E-7|
|2025-12-01 06:00:00|1000.0        |6.0     |-69.75   |1.3442016E-7 |
|2025-12-01 06:00:00|1000.0        |6.0     |-69.0    |1.5200138E-7 |
|2025-12-01 06:00:00|1000.0        |6.0     |-68.25   |1.2805143E-7 |
|2025-12-01 06:00:0

In [ ]:
# Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)

M_AR = 28.9644 # g/mol
M_CO = 28.0101 # g/mol
FATOR_CONVERSAO = (M_AR / M_CO) * 1e9  # ~ 1.03407e9

drop_cols = ["valid_time", "pressure_level", "co"]

df_monoxido_carbono_ppb = \
    (df_monoxido_carbono
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("Monóxido de carbono") 
                     ,"valor": (F.col("co") * F.lit(FATOR_CONVERSAO)).cast("double")
                     ,"unidade_medida": F.lit("ppb")})
         .drop(*drop_cols)

    )

+--------+---------+------------+-------------------+------------------+--------------+
|latitude|longitude|data_medicao|          indicador|             valor|unidade_medida|
+--------+---------+------------+-------------------+------------------+--------------+
|     6.0|    -73.5|  2025-12-01|Monóxido de carbono| 252.5471714069002|           ppb|
|     6.0|   -72.75|  2025-12-01|Monóxido de carbono|  216.060033358055|           ppb|
|     6.0|    -72.0|  2025-12-01|Monóxido de carbono| 166.4764615272159|           ppb|
|     6.0|   -71.25|  2025-12-01|Monóxido de carbono|141.79024460998514|           ppb|
|     6.0|    -70.5|  2025-12-01|Monóxido de carbono|122.27174330572366|           ppb|
+--------+---------+------------+-------------------+------------------+--------------+
only showing top 5 rows


In [22]:
# df_monoxido_carbono_ppb.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.csv", index=False)

df_monoxido_carbono_ppb.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.parquet")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [23]:
df_monoxido_carbono_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.parquet")

df_monoxido_carbono_parquet.printSchema()
df_monoxido_carbono_parquet.show(10, False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+--------+---------+------------+-------------------+------------------+--------------+
|latitude|longitude|data_medicao|indicador          |valor             |unidade_medida|
+--------+---------+------------+-------------------+------------------+--------------+
|6.0     |-73.5    |2025-12-01  |Monóxido de carbono|252.5471714069002 |ppb           |
|6.0     |-72.75   |2025-12-01  |Monóxido de carbono|216.060033358055  |ppb           |
|6.0     |-72.0    |2025-12-01  |Monóxido de carbono|166.4764615272159 |ppb           |
|6.0     |-71.25   |2025-12-01  |Monóxido de carbono|141.79024460998514|ppb           |
|6.0     |-70.5    |2025-12-01  |Monóxido de carbono|122.27174330572366|ppb           |
|6.0     |-69.75   |2025-12-01  |Monóxido d